In [1]:
import pandas as pd

df=pd.read_csv("../data/preprocessed/processed_data.csv")
df.head()

,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,CustomerTenureGroup
0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0,1,29.85,29.85,0,14.925000,0
1,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0,34,56.95,1889.50,0,53.985714,2
2,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0,2,53.85,108.15,1,36.050000,0
3,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0,45,42.30,1840.75,0,40.016304,2
4,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0,2,70.70,151.65,1,50.550000,0


In [2]:
df.shape

(7043, 48)

In [3]:
X= df.drop(columns=['Churn'])
y= df['Churn']

In [4]:
print(X.shape)
print(y.shape)
print(y.value_counts())

(7043, 47)
(7043,)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X,y,random_state=42, test_size=0.2,stratify=y)

imbalance handling

In [6]:
from imblearn.over_sampling import SMOTE

smote= SMOTE(random_state=42)

X_train_smote, y_train_smote= smote.fit_resample(X_train, y_train)

In [7]:
print(y_train_smote.value_counts())

Churn
0    4139
1    4139
Name: count, dtype: int64


scaling

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_smote)

X_test_scaled = scaler.transform(X_test)

LOGISTIC REGRESSION

In [9]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train_smote)

lr_pred = lr.predict(X_test_scaled)

RANDOM FOREST

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test)

XGBOOST

In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train_smote, y_train_smote)

xgb_pred = xgb.predict(X_test)

EVALUATION

In [12]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

def evaluate_model(name,y_true,y_pred):

    return {

        "Model":name,

        "Accuracy":accuracy_score(y_true, y_pred),

        "Precision":precision_score(y_true, y_pred),

        "Recall":recall_score(y_true, y_pred),

        "F1":f1_score(y_true, y_pred)

    }

In [13]:
results = []

results.append(
    evaluate_model("Logistic Regression", y_test, lr_pred)
)

results.append(
    evaluate_model("Random Forest", y_test, rf_pred)
)

results.append(
    evaluate_model("XGBoost", y_test, xgb_pred)
)

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.731725,0.496454,0.748663,0.597015
1,Random Forest,0.771469,0.580247,0.502674,0.538682
2,XGBoost,0.792761,0.620588,0.564171,0.591036


In [14]:
from sklearn.metrics import roc_auc_score

lr_prob = lr.predict_proba(X_test_scaled)[:,1]

lr_auc = roc_auc_score(y_test, lr_prob)

print("LR ROC AUC:",lr_auc)

LR ROC AUC: 0.8310470433232582


In [15]:
rf_prob = rf.predict_proba(X_test)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("RF ROC AUC:",rf_auc)

RF ROC AUC: 0.8146193391717688


In [16]:
xgb_prob = xgb.predict_proba(X_test)[:,1]

xgb_auc = roc_auc_score(y_test, xgb_prob)

print("XGB ROC AUC:",xgb_auc)

XGB ROC AUC: 0.8371257330336614


HYPERPARAMETER TUNING(gridsearchcv)

linear regression

In [17]:
from sklearn.model_selection import GridSearchCV

lr_params = {

    "C":[0.01,0.1,1,10],
    "penalty":["l1","l2"],
    "solver":["liblinear"]
}

lr_grid = GridSearchCV(
    
    LogisticRegression(max_iter=1000),
    lr_params,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

lr_grid.fit(X_train_scaled, y_train_smote)

best_lr = lr_grid.best_estimator_
print(lr_grid.best_params_)

best_lr_pred= best_lr.predict(X_test_scaled)

{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}


C:\Users\lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


xgboost


In [18]:
xgb_params = {

    "n_estimators":[100,200],
    "max_depth":[4,6,8],
    "learning_rate":[0.01,0.1],
    "subsample":[0.8,1]
}

xgb_grid = GridSearchCV(

    estimator=xgb,
    param_grid=xgb_params,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

xgb_grid.fit(X_train_smote, y_train_smote)

best_xgb = xgb_grid.best_estimator_

print(xgb_grid.best_params_)

best_xgb_pred= best_xgb.predict(X_test)

{'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 100, 'subsample': 1}


COMPARING TUNED MODELS


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_lr_metrics = {

    "Model":"Baseline LR",
    
    "Accuracy":
    accuracy_score(y_test,lr_pred),

    "Precision":
    precision_score(y_test,lr_pred),

    "Recall":
    recall_score(y_test,lr_pred),

    "F1":
    f1_score(y_test,lr_pred),

    "ROC_AUC":
    roc_auc_score(y_test,lr.predict_proba(X_test_scaled)[:,1])

}

tuned_lr_metrics = {

    "Model":"Tuned LR",
    
    "Accuracy":
    accuracy_score(y_test,best_lr_pred),

    "Precision":
    precision_score(y_test,best_lr_pred),

    "Recall":
    recall_score(y_test,best_lr_pred),

    "F1":
    f1_score(y_test,best_lr_pred),

    "ROC_AUC":
    roc_auc_score(y_test,best_lr.predict_proba(X_test_scaled)[:,1])

}

comparison_df = pd.DataFrame([baseline_lr_metrics,tuned_lr_metrics])

comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline LR,0.731725,0.496454,0.748663,0.597015,0.831047
1,Tuned LR,0.729595,0.493761,0.740642,0.592513,0.829430


In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_xgb_metrics = {

    "Model":"Baseline XGB",
    "Accuracy":
    accuracy_score(y_test,xgb_pred),

    "Precision":
    precision_score(y_test,xgb_pred),

    "Recall":
    recall_score(y_test,xgb_pred),

    "F1":
    f1_score(y_test,xgb_pred),

    "ROC_AUC":
    roc_auc_score(y_test,xgb.predict_proba(X_test)[:,1])

}

tuned_xgb_metrics = {

    "Model":"Tuned XGB",
    "Accuracy":
    accuracy_score(y_test,best_xgb_pred),

    "Precision":
    precision_score(y_test,best_xgb_pred),

    "Recall":
    recall_score(y_test,best_xgb_pred),

    "F1":
    f1_score(y_test,best_xgb_pred),

    "ROC_AUC":
    roc_auc_score(y_test,best_xgb.predict_proba(X_test)[:,1])

}

comparison_df = pd.DataFrame([baseline_xgb_metrics,tuned_xgb_metrics])

comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline XGB,0.792761,0.620588,0.564171,0.591036,0.837126
1,Tuned XGB,0.781405,0.581683,0.628342,0.604113,0.831103


In [21]:
tuned_comparison_df=pd.DataFrame([baseline_lr_metrics,tuned_lr_metrics,baseline_xgb_metrics,tuned_xgb_metrics])

tuned_comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline LR,0.731725,0.496454,0.748663,0.597015,0.831047
1,Tuned LR,0.729595,0.493761,0.740642,0.592513,0.829430
2,Baseline XGB,0.792761,0.620588,0.564171,0.591036,0.837126
3,Tuned XGB,0.781405,0.581683,0.628342,0.604113,0.831103


In [24]:
#k fold cv
from sklearn.model_selection import cross_val_score
tuned_xgb_scores = cross_val_score(
    best_xgb,
    X_train_smote,
    y_train_smote,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print(tuned_xgb_scores)

print(tuned_xgb_scores.mean())

[0.68257411 0.73627119 0.90806543 0.90286356 0.90222222]
0.8263993017074173


GRAPHICAL COMPARISON

In [ ]:
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

plot_df = tuned_comparison_df.set_index("Model")[metrics]

plot_df.plot(
    kind="bar",
    figsize=(12,6)
)

plt.title("Model Performance Comparison")

plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05,1))

plt.tight_layout()
plt.savefig("../images/model_comparison_bar.png", dpi=320, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

lr_probs = best_lr.predict_proba(X_test_scaled)[:,1]

xgb_probs = best_xgb.predict_proba(X_test)[:,1]

lr_fpr,lr_tpr,_ = roc_curve(y_test,lr_probs)

xgb_fpr,xgb_tpr,_ = roc_curve(y_test, xgb_probs)

plt.figure(figsize=(8,6))

plt.plot(lr_fpr,lr_tpr,label="Logistic Regression")

plt.plot(xgb_fpr, xgb_tpr, label="XGBoost")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()
plt.savefig("../images/model_comparison_roc.png", dpi=320, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

lr_cm = confusion_matrix(y_test,best_lr_pred)

xgb_cm = confusion_matrix(y_test,best_xgb_pred)

ConfusionMatrixDisplay(lr_cm).plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

ConfusionMatrixDisplay(xgb_cm).plot()
plt.title( "XGBoost Confusion Matrix")
plt.savefig("../images/model_comparison_CM.png", dpi=320, bbox_inches="tight")
plt.show()

SHAP EXPLAINABILITY

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(best_xgb)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test
)

SAVING MODEL

In [ ]:
import pickle

with open("../models/churn_model.pkl", "wb") as f:
    pickle.dump(best_xgb,f)

In [ ]:
with open("../models/scaler.pkl","wb") as f:
    pickle.dump(scaler, f)

In [ ]:
feature_columns = list(X.columns)

with open("../models/feature_columns.pkl","wb") as f:
    pickle.dump(feature_columns,f)